In [8]:
from ultralytics import YOLO
import cv2

In [1]:
from pathlib import Path
import shutil

In [4]:
BASE_DIR = Path('B:/Datasets/face mask')
CLASSES = ['with_mask', 'without_mask', 'mask_weared_incorrect']

In [11]:
from sklearn.model_selection import train_test_split
import xmltodict

annotations_dir = BASE_DIR / 'annotations'
annotations = [annotations_dir for annotations_dir in annotations_dir.iterdir()]
train_annotations, val_annotations = train_test_split(
    [xmltodict.parse(annotation.read_text()) for annotation in annotations], 
    test_size=0.2,
    random_state=42
)

In [13]:
def create_markup_string(obj, size) -> str:
    x_center = ((int(obj['bndbox']['xmin']) + int(obj['bndbox']['xmax'])) / 2) / int(size['width'])
    y_center = ((int(obj['bndbox']['ymin']) + int(obj['bndbox']['ymax'])) / 2) / int(size['height'])
    width = ((int(obj['bndbox']['xmax']) - int(obj['bndbox']['xmin']))) / int(size['width'])
    height = ((int(obj['bndbox']['ymax']) - int(obj['bndbox']['ymin']))) / int(size['height'])
    return f"{CLASSES.index(obj['name'])} {round(x_center,3)} {round(y_center,3)} {round(width,3)} {round(height,3)}\n"

In [14]:
def prepare_data(annotations, mode):
    data_dir = BASE_DIR / mode
    old_images_dir = BASE_DIR / 'images'
    images_dir = data_dir / 'images'
    labels_dir = data_dir / 'labels'
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for annotation in annotations:
        #если объктов несколько
        text_label = ''
        if isinstance(annotation['annotation']['object'],list):
            for obj in annotation['annotation']['object']:
                text_label += create_markup_string(obj, annotation['annotation']['size'])
        else:
            text_label += create_markup_string(annotation['annotation']['object'], annotation['annotation']['size'])
        
        #копирование картинки и файла разметки
        shutil.copy(old_images_dir / annotation['annotation']['filename'], 
                    images_dir / annotation['annotation']['filename'])
        
        with (labels_dir / annotation['annotation']['filename'].replace('.png','.txt')).open("w", encoding ="utf-8") as f:
            f.write(text_label)


prepare_data(train_annotations, 'train')
prepare_data(val_annotations, 'val')



In [15]:
with (BASE_DIR / 'data.yaml').open("w", encoding ="utf-8") as f:
            f.write(f'''
path: {BASE_DIR}
# Имена подпапок
train: train/images
val: val/images

# Число классов
nc: 3

names:
  0: {CLASSES[0]}
  1: {CLASSES[1]}
  2: {CLASSES[2]}          

''')

In [6]:
model_dir = Path.cwd().parent / 'models'
model_dir

WindowsPath('c:/Users/ASUS/Documents/GitHub/CV-training/models')

In [7]:
model = YOLO(model_dir / "best.pt")

In [40]:
def detect(frame):
    results = model(frame, iou = 0.8, conf = 0.5)
    frame_after = frame.copy()
    for i,result in enumerate (results):
        print(i,result)
        xyxy = result.boxes.xyxy
        names = [result.names[cls.item()] for cls in result.boxes.cls.int()]
        for name, [x1,y1,x2,y2] in zip(names,xyxy):
            cv2.rectangle(frame_after, (int(x1),int(y1)), (int(x2),int(y2)), (255, 0, 0), 3)
            cv2.putText(frame_after, name, (int(x1) + 5, int(y2) - 5), 0 ,0.4,(255,0,0), 1)
    return frame_after

In [ ]:
import time

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print('Камера недоступна')
    

ret, frame = cap.read()

time_t = time.time()

frame_after = frame.copy()
while cap.isOpened():
    ret, frame = cap.read()
    frame = cv2.resize(frame, (640, 480))

    if not ret:
            break
    if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    if cv2.waitKey(1) & 0xFF == ord('z'):
        time_t = time.time()
        
        results = model(frame, iou = 0.8, conf = 0.5)
        frame_after = detect(frame)
        
        cv2.imshow('frame_after', frame_after)
    cv2.imshow('frame', frame)
        
        
cap.release()
            
cv2.destroyAllWindows()



0: 480x640 1 with_mask, 75.4ms
Speed: 2.1ms preprocess, 75.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
0 ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'with_mask', 1: 'without_mask', 2: 'mask_weared_incorrect'}
obb: None
orig_img: array([[[252, 254, 255],
        [252, 254, 255],
        [252, 254, 255],
        ...,
        [253, 255, 255],
        [253, 255, 255],
        [253, 255, 255]],

       [[252, 254, 255],
        [252, 254, 255],
        [252, 254, 255],
        ...,
        [253, 255, 255],
        [253, 255, 255],
        [253, 255, 255]],

       [[252, 254, 255],
        [252, 254, 255],
        [252, 254, 255],
        ...,
        [253, 255, 255],
        [253, 255, 255],
        [253, 255, 255]],

       ...,

       [[229, 230, 243],
        [232, 233, 246],
        [235, 237, 249],
        ...,
        [253, 255, 255],
        [253, 255, 255

WindowsPath('C:/Users/ASUS/Downloads/фото.jpg')

In [50]:
img = cv2.imread(str(Path("C:/Users/ASUS/Downloads/фото.jpg")))
img = detect(img)
cv2.imshow('frame', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

0: 480x640 3 without_masks, 58.9ms
Speed: 2.5ms preprocess, 58.9ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)
0 ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'with_mask', 1: 'without_mask', 2: 'mask_weared_incorrect'}
obb: None
orig_img: array([[[196, 200, 201],
        [195, 199, 200],
        [194, 198, 199],
        ...,
        [ 66,  62,  61],
        [ 75,  67,  67],
        [ 81,  74,  71]],

       [[200, 204, 205],
        [197, 201, 202],
        [194, 198, 199],
        ...,
        [ 67,  63,  62],
        [ 71,  63,  63],
        [ 72,  65,  62]],

       [[203, 207, 208],
        [199, 203, 204],
        [195, 199, 200],
        ...,
        [ 69,  65,  64],
        [ 73,  65,  65],
        [ 75,  68,  65]],

       ...,

       [[197, 192, 194],
        [203, 198, 200],
        [207, 202, 204],
        ...,
        [159, 170, 192],
        [161, 172, 